Use this jupyter Notebook to try out code locally.

In [1]:
import fastf1

In [ ]:
import pandas as pd

# Get data from Silverstone for the last 15 years (2011-2026)
silverstone_data_ferrari = []

for year in range(2011, 2026):
    try:
        # Get the Silverstone race session for each year
        # Silverstone is usually round 10
        session = fastf1.get_session(year, 'Silverstone', 'R')
        session.load()
        
        # Get lap data and filter for Ferrari team
        ferrari_laps = session.laps.pick_teams('Ferrari')
        
        # Add year column for reference
        ferrari_laps = ferrari_laps.copy()
        ferrari_laps['Year'] = year
        
        silverstone_data_ferrari.append(ferrari_laps)
        print(f"Loaded {len(ferrari_laps)} Ferrari laps from Silverstone {year}")
    except Exception as e:
        print(f"Could not load Silverstone {year}: {e}")

# Combine all data into a single dataframe
df = pd.concat(silverstone_data_ferrari, ignore_index=True)
print(f"\nTotal Ferrari laps at Silverstone (2011-2026): {len(df)}")
df.head()

NotADirectoryError: Cache directory does not exist! Please check for typos or create it first.

In [8]:
import fastf1
import sqlite3
import pandas as pd

# Cache für schnellere Ladezeiten
fastf1.Cache.enable_cache('f1_cache') 

# Eure Auswahl für hohe Datenkonsistenz
TOP_TEAMS = ['Ferrari', 'Mercedes', 'Red Bull Racing', 'McLaren', 'Williams']
LEGACY_TRACKS = ['Silverstone', 'Monaco', 'Monza', 'Spa-Francorchamps', 'Barcelona', 'Interlagos', 'Spielberg']

def load_legacy_data(years):
    conn = sqlite3.connect('f1_project.db')
    
    for year in years:
        # Den Rennkalender für das Jahr abrufen
        schedule = fastf1.get_event_schedule(year)
        
        # Nur Rennen filtern, die in unserer Legacy-Liste sind
        # Wir nutzen 'str.contains', da die Namen manchmal leicht variieren (z.B. "British Grand Prix")
        selected_events = schedule[schedule['EventName'].str.contains('|'.join(LEGACY_TRACKS), case=False)]
        
        for _, event in selected_events.iterrows():
            round_num = event['RoundNumber']
            try:
                print(f"Lade {year} {event['EventName']}...")
                session = fastf1.get_session(year, round_num, 'R')
                session.load(laps=True, weather=True)
                
                laps = session.laps
                
                # Filter 1: Top Teams
                laps = laps[laps['Team'].isin(TOP_TEAMS)].copy()
                if laps.empty: continue

                # Wetter extrahieren
                weather = session.weather_data
                avg_air_temp = weather['AirTemp'].mean()
                is_raining = 1 if weather['Rainfall'].any() else 0

                # Feature Engineering
                laps['LapTimeSec'] = laps['LapTime'].dt.total_seconds()
                laps['Track'] = event['EventName']
                laps['AirTemp'] = avg_air_temp
                laps['IsRaining'] = is_raining
                laps['IsOutlap'] = laps['PitOutTime'].notnull().astype(int)
                laps['IsPitstop'] = laps['PitInTime'].notnull().astype(int)
                laps['IsClean'] = (laps['TrackStatus'] == '1').astype(int)
                
                # Wichtig: Wir speichern nur Runden mit gültiger Zeit (keine Ausfälle)
                laps = laps.dropna(subset=['LapTimeSec'])

                # In DB speichern
                laps[['Year', 'Track', 'Team', 'Driver', 'LapNumber', 'LapTimeSec', 
                      'Compound', 'TyreLife', 'AirTemp', 'IsRaining', 
                      'IsOutlap', 'IsPitstop', 'IsClean']].to_sql('laptimes', conn, if_exists='append', index=False)
                
            except Exception as e:
                print(f"Fehler bei {year} Runde {round_num}: {e}")

    conn.close()
    print("Download beendet.")

# Start: z.B. die letzten 10 Jahre
load_legacy_data(range(2014, 2024))

NotADirectoryError: Cache directory does not exist! Please check for typos or create it first.